# 02 - Stacking Classifier: Generalización Apilada y Meta-Aprendizaje Out-of-Fold

## Curso de Machine Learning: Modelos de Clasificación

Mientras que `VotingClassifier` agrega las predicciones usando una regla fija (mayoría o promedio ponderado manual), **Stacking (*Stacked Generalization*)**, propuesto por David Wolpert (1992), da un paso más allá: **entrena un modelo supervisor adicional (llamado Meta-Clasificador o `final_estimator` de Nivel 1)** cuyas variables de entrada son las probabilidades predichas por los clasificadores base (Nivel 0).

---

### Contenidos
1. Arquitectura de Dos Niveles en *Stacked Generalization* (Nivel 0 y Nivel 1)
2. El Peligro Crítico del *Data Leakage* en Stacking y la Solución **Out-of-Fold (`cross_val_predict`)**
3. Configuración de `StackingClassifier` en Scikit-Learn (`cv`, `stack_method`, `passthrough`)
4. Inspección de los Coeficientes del Meta-Clasificador (¿A qué modelo base le cree más?)
5. Stacking Multicapa (Nivel 0 -> Nivel 1 -> Nivel 2)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. ¿Cómo evita `StackingClassifier` el Sobreajuste? Predicciones Out-of-Fold

**Error común**: Si entrenamos los modelos base de Nivel 0 con todo `X_train` y usamos sus predicciones sobre ese mismo `X_train` para entrenar al Meta-Clasificador, un modelo que memorice el entrenamiento (ej. un árbol sin poda con $100\%$ de exactitud en `X_train`) engañará al Meta-Clasificador haciéndole creer que es perfecto.

**La solución rigurosa (Out-of-Fold Predictions)**:
1. Se divide `X_train` en $K$ particiones (*K-Fold Cross-Validation*).
2. Para cada partición $k$, los modelos de Nivel 0 se entrenan en las $K-1$ particiones restantes y predicen sobre la partición $k$ no vista.
3. Al concatenar las predicciones *Out-of-Fold* de las $K$ particiones, obtenemos una matriz de **meta-características** $\mathbf{Z}_{\text{train}} \in \mathbb{R}^{N \times M}$ limpia de fuga de datos (*data leakage*).
4. El `final_estimator` (típicamente una `LogisticRegression` regularizada) se entrena sobre $\mathbf{Z}_{\text{train}}$ para predecir $y_{\text{train}}$.


In [ ]:
# Construccion de StackingClassifier en el dataset Cancer de Mama
cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target)

nivel_0 = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('et', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42)),
    ('svc', make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42))),
    ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)))
]

# Meta-Clasificador de Nivel 1: Regresion Logistica penalizada L2
meta_learner = LogisticRegression(C=1.0, random_state=42)

stack_clf = StackingClassifier(
    estimators=nivel_0,
    final_estimator=meta_learner,
    cv=5,                     # 5-Fold Out-of-Fold predictions
    stack_method='predict_proba',
    passthrough=False,        # Si es True, concatena X original con las predicciones de Nivel 0
    n_jobs=-1
)
stack_clf.fit(X_tr, y_tr)

preds_stack = stack_clf.predict(X_te)
probs_stack = stack_clf.predict_proba(X_te)[:, 1]

print(f"Exactitud de StackingClassifier en Test: {accuracy_score(y_te, preds_stack)*100:.2f}%")
print(f"ROC-AUC de StackingClassifier en Test:   {roc_auc_score(y_te, probs_stack):.4f}")


In [ ]:
# Inspeccion de los pesos (coeficientes) asignados por el Meta-Clasificador a cada modelo de Nivel 0
nombres_modelos = [nombre for nombre, _ in nivel_0]
pesos_meta = stack_clf.final_estimator_.coef_[0]

df_pesos = pd.DataFrame({'Modelo Base (Nivel 0)': nombres_modelos, 'Coeficiente Meta-Modelo (Nivel 1)': pesos_meta})
df_pesos = df_pesos.sort_values(by='Coeficiente Meta-Modelo (Nivel 1)', ascending=True)

plt.figure(figsize=(8, 4.2))
plt.barh(df_pesos['Modelo Base (Nivel 0)'], df_pesos['Coeficiente Meta-Modelo (Nivel 1)'], color='#2b5c8f')
plt.xlabel('Peso Aprendido por el Meta-Clasificador (LogisticRegression)', fontweight='bold')
plt.title('Importancia Asignada por el Meta-Clasificador a cada Modelo de Nivel 0', fontweight='bold')
plt.tight_layout()
plt.show()
